In [ ]:
#from pyspark import SparkContext #stara wersja

In [1]:
from pyspark.sql import SparkSession

In [ ]:
#spark = SparkSession.builder.getOrCreate()

In [2]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

In [3]:
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
#2.1 - agregat po sklepie
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
#2.2 Statystyka per kategoria

from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

category_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN")
    )
    .orderBy("category")
)

category_summary.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [7]:
#3.1 - grupowania po przedziale czasowym w oknach
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [8]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [9]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [11]:
#3.2 - Okna 30-minutowe per sklep
half_hourly_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store") 

In [12]:
(
    half_hourly_store
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .show(truncate=False)
)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [13]:
#3.3 — W której godzinie sklep “Kraków” miał najwyższy przychód

from pyspark.sql.functions import desc

krakow_best_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_PLN"))
    .orderBy(desc("suma_PLN"))
)


In [14]:
(
    krakow_best_hour
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "suma_PLN"
    )
    .show(1, truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
+-------------------+-------------------+---------+
only showing top 1 row



In [10]:
#4.1 Okno 1h, krok 30 minut
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [16]:
#4.2 - Porównaj tumbling vs sliding
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Okno przesuwne (sliding) generuje więcej wierszy wynikowych (sliding- 7 okien; tumbling - 1 okno), ponieważ przedziały czasowe nakładają się na siebie. 
# W tym przypadku (okno 1h, krok 30min), jedna transakcja (np. z godziny 09:15) znajduje się jednocześnie w dwóch różnych oknach: 
# - OKNO1: 08:30 - 09:30
# - OKNO2: 09:00 - 10:00
# okna tumbling nie nakładają się na siebie zatem jest ich mniej.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


Odpowiedzi na pytania: 

1. Ile transakcji jest w oknie 9:00-10:00?
    odp: 4661 transakcji

2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
    odp: groupBy("store") zlicza dane globalnie dla całego zbioru, a groupBy(window(..), "store") robi agregację osobno wewnątrz każdego okna, efektem czego jest przedstawienei tego jak sprzedaż zmieniała się godzina po godzinie.

3.  W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
    odp: Zakładając, że przedziały z punktu 4.1 są domknięte, transakcje z godziny 9:30 będą zawierać się w 3 oknach:

    - 8:30 - 9:30
    - 9:00 - 10:00
    - 9:30 - 10:30

    W Sparku domyślnie przedziały są lewostronnie domknięte, zatem transakcje z godziny 9:30 będą zawierać się w 2 oknach:

    - 8:30 - 9:30
    - 9:30 - 10:30

In [18]:
#Praca domowa
#1.
from pyspark.sql.functions import avg, desc, asc

(
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia"))
    .orderBy(asc("srednia"))
    .select(col("window.start").alias("od"), col("window.end").alias("do"), "srednia")
    .show(1, truncate=False)
)

+-------------------+-------------------+-------+
|od                 |do                 |srednia|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01 |
+-------------------+-------------------+-------+
only showing top 1 row



In [20]:
### 2. 
(
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .filter(col("window.start").cast("string").like("% 09:00:%"))
    .select("category", "liczba_tx")
    .show(truncate=False)
)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|odzież     |605      |
|książki    |622      |
|żywność    |567      |
+-----------+---------+



In [21]:
(
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx")) 
    .select(col("window.start").alias("od"), col("window.end").alias("do"), "liczba_tx")
    .show(1, truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row



In [ ]:
spark.stop()